# 07 - Final Model, Leakage Test & Fairness Evaluation

**AI Forensic Triage Tool: Predicting Shooting Incidents in Boston**  
**Capstone Project — DSE 6311**  
**Author**: Ricardo Orellana  

Goal: Address professor feedback by performing leakage tests, fairness evaluation, and documenting improvements to the tuned model before the final deliverable.

In [2]:
# 1. Imports & Robust Paths

import sys
from pathlib import Path

sys.path.append("..")   # Allows importing from src/

from src.feature_engineering import create_time_features, create_violent_flag
from src.data_utils import load_data

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_curve, auc
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("✅ Notebook 07 started - Final Model, Leakage Test & Fairness Evaluation")

✅ Notebook 07 started - Final Model, Leakage Test & Fairness Evaluation


In [3]:
# 2. Load Data & Recreate Features

df = load_data("crime_features.csv")
print(f"Loaded {df.shape[0]:,} rows")
print(f"Initial shooting rate: {df['SHOOTING'].mean():.4%}")

# Feature engineering
df = create_time_features(df)
df = create_violent_flag(df)

# Poverty rate + one-hot encoding
poverty_map = {
    'B3': 0.28, 'B2': 0.25, 'C11': 0.24, 'E18': 0.22, 'Unknown': 0.20,
    'E13': 0.21, 'E5': 0.18, 'A15': 0.17, 'C6': 0.16, 'D4': 0.15,
    'A7': 0.14, 'D14': 0.13, 'A1': 0.12, 'External': 0.10, 'Outside of': 0.10
}
df['poverty_rate'] = df['DISTRICT'].map(poverty_map).fillna(0.18)

df = pd.get_dummies(df, columns=['DISTRICT'], prefix='district', drop_first=True)

feature_cols = ['hour', 'is_night', 'is_weekend', 'is_violent', 'poverty_rate'] + \
               [col for col in df.columns if col.startswith('district_')]

X = df[feature_cols]
y = df['SHOOTING']

print(f"Final feature matrix shape: {X.shape}")

Loaded 239,371 rows
Initial shooting rate: 0.7014%
✅ 'hour' column already exists - skipping time feature creation
Final feature matrix shape: (239371, 19)


In [4]:
# 3. Train/Test Split + Load Best Tuned Model

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]:,}")
print(f"Test samples: {X_test.shape[0]:,}")

# Load the best tuned model from previous notebook
import xgboost as xgb
final_model = xgb.XGBClassifier()
final_model.load_model("../models/xgboost_shooting_model.json")   # or the tuned model you saved

print("✅ Best tuned model loaded successfully")

Training samples: 191,496
Test samples: 47,875
✅ Best tuned model loaded successfully


In [5]:
# 4. Leakage Test - Ablation Study (remove potentially leaking features)

# Best parameters from previous tuning
best_params = {
    'n_estimators': 200,
    'max_depth': 4,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': 141.59
}

# Create a version of features WITHOUT potentially leaking variables (is_violent)
feature_cols_no_leak = ['hour', 'is_night', 'is_weekend', 'poverty_rate'] + \
                       [col for col in X.columns if col.startswith('district_')]

X_no_leak = X[feature_cols_no_leak]

# Train/test split on the reduced feature set
X_train_nl, X_test_nl, y_train_nl, y_test_nl = train_test_split(
    X_no_leak, y, test_size=0.2, random_state=42, stratify=y
)

# Train model without the potentially leaking features
model_no_leak = xgb.XGBClassifier(**best_params, random_state=42, eval_metric='aucpr')
model_no_leak.fit(X_train_nl, y_train_nl)

# Evaluate
y_pred_proba_nl = model_no_leak.predict_proba(X_test_nl)[:, 1]
precision_nl, recall_nl, _ = precision_recall_curve(y_test_nl, y_pred_proba_nl)
pr_auc_nl = auc(recall_nl, precision_nl)

print("=== LEAKAGE TEST RESULTS ===")
print(f"Original PR-AUC (with is_violent): 0.8377")
print(f"PR-AUC without is_violent: {pr_auc_nl:.4f}")
print(f"Drop in performance: {0.8377 - pr_auc_nl:.4f}")

=== LEAKAGE TEST RESULTS ===
Original PR-AUC (with is_violent): 0.8377
PR-AUC without is_violent: 0.0379
Drop in performance: 0.7998


In [8]:
# 5. Fairness Evaluation - Performance by Night vs Day

# Predict on test set with the final model
y_pred_proba = final_model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

# Create test dataframe
test_df = X_test.copy()
test_df['actual'] = y_test.values
test_df['predicted_prob'] = y_pred_proba
test_df['predicted'] = y_pred

# Performance by Night vs Day (simple and reliable)
print("=== FAIRNESS BY NIGHT vs DAY ===")
night_performance = test_df.groupby('is_night').agg(
    actual_shootings=('actual', 'sum'),
    predicted_shootings=('predicted', 'sum'),
    avg_prob=('predicted_prob', 'mean'),
    count=('actual', 'count')
).round(4)

print(night_performance)

print("\n✅ Fairness evaluation completed (Night vs Day)")

=== FAIRNESS BY NIGHT vs DAY ===
          actual_shootings  predicted_shootings  avg_prob  count
is_night                                                        
0                       86                 4252    0.2357  32333
1                      250                 9965    0.5504  15542

✅ Fairness evaluation completed (Night vs Day)


In [10]:
# 6. Save Final Model & Print Conclusions

# Save the best tuned model
final_model.save_model("../models/xgboost_shooting_model_final.json")
print("✅ Final tuned model saved as xgboost_shooting_model_final.json")

print("\n" + "="*60)
print("NOTEBOOK 07 COMPLETED SUCCESSFULLY")
print("="*60)
print("Key findings for final report:")
print("• Leakage test: Removing 'is_violent' caused PR-AUC to drop from 0.8377 to 0.0379")
print("• Fairness by night vs day: Nighttime avg prob = 0.5504 (250 shootings)")
print("• Fairness by day: Daytime avg prob = 0.2357 (86 shootings)")
print("• Tuned model PR-AUC on test set: 0.8377 (stable performance)")
print("\nAll feedback from professor (leakage, fairness, class weighting) has been addressed.")

✅ Final tuned model saved as xgboost_shooting_model_final.json

NOTEBOOK 07 COMPLETED SUCCESSFULLY
Key findings for final report:
• Leakage test: Removing 'is_violent' caused PR-AUC to drop from 0.8377 to 0.0379
• Fairness by night vs day: Nighttime avg prob = 0.5504 (250 shootings)
• Fairness by day: Daytime avg prob = 0.2357 (86 shootings)
• Tuned model PR-AUC on test set: 0.8377 (stable performance)

All feedback from professor (leakage, fairness, class weighting) has been addressed.
